# Building a Bilingual AI Application (构建双语AI应用)

## Overview | 概述

This notebook demonstrates how to build a production-ready bilingual (Chinese-English) AI application with the following features:

本笔记本演示如何构建一个生产级的双语（中英文）AI应用，具有以下功能：

1. **Model Management** - Fetch and cache available OpenAI models
   - **模型管理** - 获取并缓存可用的OpenAI模型

2. **Parameter Recommendations** - Predefined parameters for common models
   - **参数推荐** - 为常见模型提供预定义参数

3. **Export Functionality** - Export to CSV and Excel with multiple sheets
   - **导出功能** - 导出到CSV和Excel多工作表格式

4. **Prompt Management** - Save/load prompts with bilingual support
   - **提示词管理** - 支持双语的提示词保存/加载

5. **Chat Integration** - Use saved prompts with OpenAI API
   - **聊天集成** - 使用保存的提示词调用OpenAI API

## Prerequisites | 前置要求

- Python 3.8 or higher
- OpenAI API key
- Required packages: openai, pandas, openpyxl

- Python 3.8或更高版本
- OpenAI API密钥
- 所需包：openai、pandas、openpyxl

## 1. Install Dependencies | 安装依赖

First, let's install the required packages.

首先，让我们安装所需的包。

In [ ]:
# Install required packages | 安装所需的包
# Uncomment the following line if you need to install packages | 如果需要安装包，请取消注释以下行
# !pip install openai pandas openpyxl python-dotenv

## 2. Import Libraries | 导入库

Import all necessary libraries for building the bilingual AI application.

导入构建双语AI应用所需的所有库。

In [ ]:
# Standard library imports | 标准库导入
import os
import json
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Any

# Third-party imports | 第三方库导入
import pandas as pd
from openai import OpenAI

print("All libraries imported successfully! | 所有库导入成功！")

## 3. Setup OpenAI API | 设置OpenAI API

Configure your OpenAI API key. Make sure to set the `OPENAI_API_KEY` environment variable.

配置您的OpenAI API密钥。确保设置`OPENAI_API_KEY`环境变量。

In [ ]:
# Initialize OpenAI client | 初始化OpenAI客户端
# Make sure to set OPENAI_API_KEY in your environment | 确保在环境中设置OPENAI_API_KEY

# Option 1: Use environment variable (recommended) | 选项1：使用环境变量（推荐）
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Option 2: Set directly (not recommended for production) | 选项2：直接设置（不推荐用于生产环境）
# client = OpenAI(api_key="your-api-key-here")

print("OpenAI client initialized! | OpenAI客户端已初始化！")

## 4. Prompt Manager Class | 提示词管理器类

The `PromptManager` class handles all prompt-related operations including saving, loading, and managing prompt collections.

`PromptManager`类处理所有提示词相关操作，包括保存、加载和管理提示词集合。

In [ ]:
class PromptManager:
    """
    Manages prompts with bilingual support (English and Chinese)
    管理支持双语（英文和中文）的提示词
    
    Features | 功能:
    - Save/load individual prompts | 保存/加载单个提示词
    - Import/export prompt collections | 导入/导出提示词集合
    - Metadata tracking | 元数据跟踪
    """
    
    def __init__(self, base_dir: str = "prompts"):
        """
        Initialize PromptManager | 初始化提示词管理器
        
        Args:
            base_dir: Base directory for storing prompts | 存储提示词的基础目录
        """
        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(exist_ok=True)
        self.prompts: Dict[str, Dict[str, Any]] = {}
        
    def save_prompt(self, 
                   name: str, 
                   content: str, 
                   description: str = "",
                   tags: List[str] = None) -> str:
        """
        Save a prompt to a text file | 将提示词保存到文本文件
        
        Args:
            name: Prompt name | 提示词名称
            content: Prompt content | 提示词内容
            description: Optional description | 可选描述
            tags: Optional tags for categorization | 可选分类标签
            
        Returns:
            Path to saved file | 保存文件的路径
        """
        if tags is None:
            tags = []
            
        # Create metadata | 创建元数据
        metadata = {
            "name": name,
            "description": description,
            "tags": tags,
            "created_at": datetime.now().isoformat(),
            "updated_at": datetime.now().isoformat()
        }
        
        # Save prompt content | 保存提示词内容
        prompt_file = self.base_dir / f"{name}.txt"
        with open(prompt_file, 'w', encoding='utf-8') as f:
            f.write(content)
        
        # Save metadata | 保存元数据
        metadata_file = self.base_dir / f"{name}_metadata.json"
        with open(metadata_file, 'w', encoding='utf-8') as f:
            json.dump(metadata, f, indent=2, ensure_ascii=False)
        
        # Store in memory | 存储在内存中
        self.prompts[name] = {
            "content": content,
            "metadata": metadata
        }
        
        return str(prompt_file)
    
    def load_prompt(self, name: str) -> Optional[Dict[str, Any]]:
        """
        Load a prompt from file | 从文件加载提示词
        
        Args:
            name: Prompt name | 提示词名称
            
        Returns:
            Dictionary with content and metadata | 包含内容和元数据的字典
        """
        prompt_file = self.base_dir / f"{name}.txt"
        metadata_file = self.base_dir / f"{name}_metadata.json"
        
        if not prompt_file.exists():
            print(f"Prompt '{name}' not found | 未找到提示词'{name}'")
            return None
        
        # Load content | 加载内容
        with open(prompt_file, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Load metadata | 加载元数据
        metadata = {}
        if metadata_file.exists():
            with open(metadata_file, 'r', encoding='utf-8') as f:
                metadata = json.load(f)
        
        prompt_data = {
            "content": content,
            "metadata": metadata
        }
        
        # Store in memory | 存储在内存中
        self.prompts[name] = prompt_data
        
        return prompt_data
    
    def list_prompts(self) -> List[Dict[str, Any]]:
        """
        List all available prompts | 列出所有可用提示词
        
        Returns:
            List of prompt metadata | 提示词元数据列表
        """
        prompts_list = []
        
        for prompt_file in self.base_dir.glob("*.txt"):
            name = prompt_file.stem
            metadata_file = self.base_dir / f"{name}_metadata.json"
            
            if metadata_file.exists():
                with open(metadata_file, 'r', encoding='utf-8') as f:
                    metadata = json.load(f)
                    prompts_list.append(metadata)
        
        return prompts_list
    
    def export_collection(self, output_file: str = "prompt_collection.json"):
        """
        Export all prompts to a single JSON file | 将所有提示词导出到单个JSON文件
        
        Args:
            output_file: Output file path | 输出文件路径
        """
        collection = {}
        
        for prompt_file in self.base_dir.glob("*.txt"):
            name = prompt_file.stem
            prompt_data = self.load_prompt(name)
            if prompt_data:
                collection[name] = prompt_data
        
        output_path = Path(output_file)
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(collection, f, indent=2, ensure_ascii=False)
        
        print(f"Exported {len(collection)} prompts to {output_file}")
        print(f"导出了{len(collection)}个提示词到{output_file}")
    
    def import_collection(self, input_file: str):
        """
        Import prompts from a JSON file | 从JSON文件导入提示词
        
        Args:
            input_file: Input file path | 输入文件路径
        """
        with open(input_file, 'r', encoding='utf-8') as f:
            collection = json.load(f)
        
        imported_count = 0
        for name, prompt_data in collection.items():
            content = prompt_data.get("content", "")
            metadata = prompt_data.get("metadata", {})
            
            self.save_prompt(
                name=name,
                content=content,
                description=metadata.get("description", ""),
                tags=metadata.get("tags", [])
            )
            imported_count += 1
        
        print(f"Imported {imported_count} prompts | 导入了{imported_count}个提示词")

print("PromptManager class defined! | PromptManager类已定义！")

## 5. Model Management Functions | 模型管理函数

Functions to fetch, cache, and export OpenAI model information.

用于获取、缓存和导出OpenAI模型信息的函数。

In [ ]:
def fetch_models(client: OpenAI, cache_file: str = "models_cache.json") -> List[Dict[str, Any]]:
    """
    Fetch available OpenAI models and cache them | 获取可用的OpenAI模型并缓存
    
    Args:
        client: OpenAI client instance | OpenAI客户端实例
        cache_file: Cache file path | 缓存文件路径
        
    Returns:
        List of model information | 模型信息列表
    """
    cache_path = Path(cache_file)
    
    # Try to load from cache | 尝试从缓存加载
    if cache_path.exists():
        try:
            with open(cache_path, 'r', encoding='utf-8') as f:
                cached_data = json.load(f)
                # Check if cache is less than 24 hours old | 检查缓存是否少于24小时
                cache_time = datetime.fromisoformat(cached_data.get("timestamp", ""))
                if (datetime.now() - cache_time).total_seconds() < 86400:
                    print("Loading models from cache | 从缓存加载模型")
                    return cached_data.get("models", [])
        except Exception as e:
            print(f"Cache read error: {e} | 缓存读取错误: {e}")
    
    # Fetch from API | 从API获取
    print("Fetching models from OpenAI API | 从OpenAI API获取模型")
    models_response = client.models.list()
    
    models = []
    for model in models_response.data:
        models.append({
            "id": model.id,
            "created": model.created,
            "owned_by": model.owned_by
        })
    
    # Cache the results | 缓存结果
    cache_data = {
        "timestamp": datetime.now().isoformat(),
        "models": models
    }
    
    with open(cache_path, 'w', encoding='utf-8') as f:
        json.dump(cache_data, f, indent=2, ensure_ascii=False)
    
    print(f"Fetched and cached {len(models)} models | 获取并缓存了{len(models)}个模型")
    return models


def get_model_parameters(model_id: str) -> Dict[str, Any]:
    """
    Get recommended parameters for a specific model | 获取特定模型的推荐参数
    
    Args:
        model_id: Model identifier | 模型标识符
        
    Returns:
        Dictionary of recommended parameters | 推荐参数字典
    """
    # Default parameters | 默认参数
    default_params = {
        "temperature": 0.7,
        "max_tokens": 1000,
        "top_p": 1.0,
        "frequency_penalty": 0.0,
        "presence_penalty": 0.0,
        "description_en": "Default parameters for general use",
        "description_zh": "通用默认参数"
    }
    
    # Model-specific recommendations | 特定模型推荐
    model_params = {
        "gpt-4": {
            "temperature": 0.7,
            "max_tokens": 2000,
            "top_p": 1.0,
            "description_en": "Optimized for complex reasoning and analysis",
            "description_zh": "优化用于复杂推理和分析"
        },
        "gpt-4-turbo": {
            "temperature": 0.7,
            "max_tokens": 4000,
            "top_p": 1.0,
            "description_en": "Faster responses with large context window",
            "description_zh": "更快的响应速度和大型上下文窗口"
        },
        "gpt-3.5-turbo": {
            "temperature": 0.7,
            "max_tokens": 1000,
            "top_p": 1.0,
            "description_en": "Cost-effective for most tasks",
            "description_zh": "适用于大多数任务的经济选择"
        }
    }
    
    # Find matching model parameters | 查找匹配的模型参数
    for key, params in model_params.items():
        if key in model_id.lower():
            return {**default_params, **params}
    
    return default_params


def export_models_to_csv(models: List[Dict[str, Any]], filename: str = "models.csv"):
    """
    Export models to CSV format | 将模型导出为CSV格式
    
    Args:
        models: List of model information | 模型信息列表
        filename: Output filename | 输出文件名
    """
    df = pd.DataFrame(models)
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"Models exported to {filename} | 模型已导出到{filename}")


def export_models_to_excel(models: List[Dict[str, Any]], filename: str = "models.xlsx"):
    """
    Export models to Excel with multiple sheets | 将模型导出为Excel多工作表格式
    
    Args:
        models: List of model information | 模型信息列表
        filename: Output filename | 输出文件名
    """
    # Create Excel writer | 创建Excel写入器
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        # Sheet 1: All models | 工作表1：所有模型
        df_all = pd.DataFrame(models)
        df_all.to_excel(writer, sheet_name='All Models | 所有模型', index=False)
        
        # Sheet 2: GPT-4 models | 工作表2：GPT-4模型
        gpt4_models = [m for m in models if 'gpt-4' in m['id'].lower()]
        if gpt4_models:
            df_gpt4 = pd.DataFrame(gpt4_models)
            df_gpt4.to_excel(writer, sheet_name='GPT-4', index=False)
        
        # Sheet 3: GPT-3.5 models | 工作表3：GPT-3.5模型
        gpt35_models = [m for m in models if 'gpt-3.5' in m['id'].lower()]
        if gpt35_models:
            df_gpt35 = pd.DataFrame(gpt35_models)
            df_gpt35.to_excel(writer, sheet_name='GPT-3.5', index=False)
        
        # Sheet 4: Model parameters | 工作表4：模型参数
        params_data = []
        for model in models[:10]:  # Limit to first 10 for example | 示例中限制为前10个
            params = get_model_parameters(model['id'])
            params_data.append({
                'Model | 模型': model['id'],
                'Temperature | 温度': params['temperature'],
                'Max Tokens | 最大令牌数': params['max_tokens'],
                'Description (EN) | 描述（英文）': params.get('description_en', ''),
                'Description (ZH) | 描述（中文）': params.get('description_zh', '')
            })
        
        df_params = pd.DataFrame(params_data)
        df_params.to_excel(writer, sheet_name='Parameters | 参数', index=False)
    
    print(f"Models exported to {filename} with multiple sheets | 模型已导出到{filename}（多工作表）")

print("Model management functions defined! | 模型管理函数已定义！")

## 6. BilingualAIApp Class | 双语AI应用类

The main application class that integrates all functionality.

集成所有功能的主应用类。

In [ ]:
class BilingualAIApp:
    """
    Main bilingual AI application class | 主要的双语AI应用类
    
    Integrates:
    - Model management | 模型管理
    - Prompt management | 提示词管理
    - Chat functionality | 聊天功能
    - Export capabilities | 导出功能
    """
    
    def __init__(self, client: OpenAI, prompts_dir: str = "prompts"):
        """
        Initialize BilingualAIApp | 初始化双语AI应用
        
        Args:
            client: OpenAI client instance | OpenAI客户端实例
            prompts_dir: Directory for storing prompts | 存储提示词的目录
        """
        self.client = client
        self.prompt_manager = PromptManager(prompts_dir)
        self.models = []
        self.conversation_history = []
        
    def initialize_models(self, cache_file: str = "models_cache.json"):
        """
        Fetch and cache available models | 获取并缓存可用模型
        
        Args:
            cache_file: Cache file path | 缓存文件路径
        """
        self.models = fetch_models(self.client, cache_file)
        print(f"Initialized with {len(self.models)} models | 已初始化{len(self.models)}个模型")
    
    def chat(self, 
            prompt: str, 
            model: str = "gpt-3.5-turbo",
            use_history: bool = False,
            **kwargs) -> str:
        """
        Send a chat message to OpenAI API | 向OpenAI API发送聊天消息
        
        Args:
            prompt: User prompt | 用户提示词
            model: Model to use | 要使用的模型
            use_history: Whether to include conversation history | 是否包含对话历史
            **kwargs: Additional parameters for API call | API调用的额外参数
            
        Returns:
            AI response | AI响应
        """
        # Get model parameters | 获取模型参数
        params = get_model_parameters(model)
        
        # Merge with user-provided kwargs | 与用户提供的kwargs合并
        api_params = {
            "temperature": kwargs.get("temperature", params["temperature"]),
            "max_tokens": kwargs.get("max_tokens", params["max_tokens"]),
            "top_p": kwargs.get("top_p", params["top_p"])
        }
        
        # Build messages | 构建消息
        messages = []
        if use_history:
            messages.extend(self.conversation_history)
        messages.append({"role": "user", "content": prompt})
        
        # Make API call | 调用API
        try:
            response = self.client.chat.completions.create(
                model=model,
                messages=messages,
                **api_params
            )
            
            assistant_message = response.choices[0].message.content
            
            # Update history | 更新历史
            if use_history:
                self.conversation_history.append({"role": "user", "content": prompt})
                self.conversation_history.append({"role": "assistant", "content": assistant_message})
            
            return assistant_message
            
        except Exception as e:
            error_msg = f"Error calling OpenAI API: {e} | 调用OpenAI API时出错: {e}"
            print(error_msg)
            return error_msg
    
    def chat_with_saved_prompt(self, 
                               prompt_name: str,
                               model: str = "gpt-3.5-turbo",
                               **kwargs) -> str:
        """
        Use a saved prompt for chat | 使用保存的提示词进行聊天
        
        Args:
            prompt_name: Name of saved prompt | 保存的提示词名称
            model: Model to use | 要使用的模型
            **kwargs: Additional parameters | 额外参数
            
        Returns:
            AI response | AI响应
        """
        prompt_data = self.prompt_manager.load_prompt(prompt_name)
        if not prompt_data:
            return f"Prompt '{prompt_name}' not found | 未找到提示词'{prompt_name}'"
        
        prompt_content = prompt_data["content"]
        return self.chat(prompt_content, model, **kwargs)
    
    def clear_history(self):
        """Clear conversation history | 清除对话历史"""
        self.conversation_history = []
        print("Conversation history cleared | 对话历史已清除")
    
    def export_models(self, format: str = "excel", filename: str = None):
        """
        Export models to file | 将模型导出到文件
        
        Args:
            format: Export format ('csv' or 'excel') | 导出格式（'csv'或'excel'）
            filename: Output filename | 输出文件名
        """
        if not self.models:
            print("No models loaded. Call initialize_models() first.")
            print("未加载模型。请先调用initialize_models()。")
            return
        
        if format.lower() == "csv":
            fname = filename or "models.csv"
            export_models_to_csv(self.models, fname)
        elif format.lower() == "excel":
            fname = filename or "models.xlsx"
            export_models_to_excel(self.models, fname)
        else:
            print(f"Unsupported format: {format} | 不支持的格式: {format}")
    
    def get_model_info(self, model_id: str) -> Optional[Dict[str, Any]]:
        """
        Get information about a specific model | 获取特定模型的信息
        
        Args:
            model_id: Model identifier | 模型标识符
            
        Returns:
            Model information dictionary | 模型信息字典
        """
        for model in self.models:
            if model['id'] == model_id:
                # Add recommended parameters | 添加推荐参数
                model['recommended_params'] = get_model_parameters(model_id)
                return model
        return None

print("BilingualAIApp class defined! | BilingualAIApp类已定义！")

## 7. Example Usage | 使用示例

Let's demonstrate the complete functionality of the bilingual AI application.

让我们演示双语AI应用的完整功能。

### 7.1 Initialize the Application | 初始化应用

In [ ]:
# Initialize the bilingual AI app | 初始化双语AI应用
app = BilingualAIApp(client, prompts_dir="example_prompts")

# Fetch and cache models | 获取并缓存模型
app.initialize_models()

print("\nApplication initialized successfully! | 应用初始化成功！")

### 7.2 Save and Manage Prompts | 保存和管理提示词

In [ ]:
# Save some example prompts | 保存一些示例提示词

# English prompt | 英文提示词
app.prompt_manager.save_prompt(
    name="code_review",
    content="You are an expert code reviewer. Please review the following code and provide suggestions for improvement.",
    description="Prompt for code review tasks",
    tags=["coding", "review"]
)

# Chinese prompt | 中文提示词
app.prompt_manager.save_prompt(
    name="translation_assistant",
    content="你是一位专业的翻译助手。请将以下内容翻译成英文，保持原意和专业术语的准确性。",
    description="用于翻译任务的提示词",
    tags=["翻译", "中英文"]
)

# Bilingual prompt | 双语提示词
app.prompt_manager.save_prompt(
    name="creative_writing",
    content="""You are a creative writing assistant. Help me write engaging content.
你是一位创意写作助手。帮助我创作引人入胜的内容。

Guidelines:
- Be creative and original | 要有创意和原创性
- Use vivid descriptions | 使用生动的描述
- Engage the reader | 吸引读者""",
    description="Creative writing assistant prompt | 创意写作助手提示词",
    tags=["writing", "creative", "bilingual"]
)

print("\nPrompts saved successfully! | 提示词保存成功！")

# List all prompts | 列出所有提示词
print("\nAvailable prompts | 可用的提示词:")
prompts_list = app.prompt_manager.list_prompts()
for prompt_meta in prompts_list:
    print(f"  - {prompt_meta['name']}: {prompt_meta.get('description', 'No description')}")

### 7.3 Export Models to Files | 导出模型到文件

In [ ]:
# Export models to CSV | 导出模型到CSV
app.export_models(format="csv", filename="openai_models.csv")

# Export models to Excel with multiple sheets | 导出模型到Excel多工作表
app.export_models(format="excel", filename="openai_models.xlsx")

print("\nModels exported successfully! | 模型导出成功！")

### 7.4 Use Saved Prompts for Chat | 使用保存的提示词进行聊天

**Note:** The following cells require a valid OpenAI API key. Make sure your key is set before running.

**注意：** 以下单元格需要有效的OpenAI API密钥。运行前请确保已设置密钥。

In [ ]:
# Example 1: Use code review prompt | 示例1：使用代码审查提示词
# Uncomment to run | 取消注释以运行

# code_to_review = """
# def calculate_sum(numbers):
#     total = 0
#     for i in range(len(numbers)):
#         total = total + numbers[i]
#     return total
# """

# Load the prompt and add the code | 加载提示词并添加代码
# prompt_data = app.prompt_manager.load_prompt("code_review")
# full_prompt = f"{prompt_data['content']}\n\n{code_to_review}"

# response = app.chat(full_prompt, model="gpt-3.5-turbo")
# print("Code Review Response | 代码审查响应:")
# print(response)

print("Uncomment the code above to test with your API key | 取消注释上面的代码以使用您的API密钥测试")

In [ ]:
# Example 2: Use translation prompt | 示例2：使用翻译提示词
# Uncomment to run | 取消注释以运行

# text_to_translate = "机器学习是人工智能的一个重要分支，它使计算机能够从数据中学习并做出预测。"

# Load the prompt and add the text | 加载提示词并添加文本
# prompt_data = app.prompt_manager.load_prompt("translation_assistant")
# full_prompt = f"{prompt_data['content']}\n\n{text_to_translate}"

# response = app.chat(full_prompt, model="gpt-3.5-turbo")
# print("Translation Response | 翻译响应:")
# print(response)

print("Uncomment the code above to test with your API key | 取消注释上面的代码以使用您的API密钥测试")

### 7.5 Export and Import Prompt Collections | 导出和导入提示词集合

In [ ]:
# Export all prompts to a collection file | 将所有提示词导出到集合文件
app.prompt_manager.export_collection("my_prompt_collection.json")

print("\nPrompt collection exported! | 提示词集合已导出！")
print("You can share this file or import it later. | 您可以共享此文件或稍后导入。")

In [ ]:
# Import prompts from a collection file | 从集合文件导入提示词
# Uncomment to test import | 取消注释以测试导入

# Create a new prompt manager for testing | 创建新的提示词管理器用于测试
# new_manager = PromptManager("imported_prompts")
# new_manager.import_collection("my_prompt_collection.json")

# List imported prompts | 列出导入的提示词
# print("\nImported prompts | 导入的提示词:")
# for prompt_meta in new_manager.list_prompts():
#     print(f"  - {prompt_meta['name']}")

print("Uncomment the code above to test prompt import | 取消注释上面的代码以测试提示词导入")

### 7.6 Get Model Information and Parameters | 获取模型信息和参数

In [ ]:
# Display information for a few models | 显示几个模型的信息
print("Model Information and Recommended Parameters | 模型信息和推荐参数\n")
print("=" * 80)

# Get some example models | 获取一些示例模型
example_model_ids = [m['id'] for m in app.models[:5]]

for model_id in example_model_ids:
    model_info = app.get_model_info(model_id)
    if model_info:
        print(f"\nModel | 模型: {model_info['id']}")
        print(f"Owned by | 所有者: {model_info['owned_by']}")
        
        params = model_info.get('recommended_params', {})
        print(f"\nRecommended Parameters | 推荐参数:")
        print(f"  Temperature | 温度: {params.get('temperature', 'N/A')}")
        print(f"  Max Tokens | 最大令牌数: {params.get('max_tokens', 'N/A')}")
        print(f"  Description (EN) | 描述（英文）: {params.get('description_en', 'N/A')}")
        print(f"  Description (ZH) | 描述（中文）: {params.get('description_zh', 'N/A')}")
        print("-" * 80)

### 7.7 Conversation with History | 带历史记录的对话

In [ ]:
# Example of maintaining conversation history | 维护对话历史的示例
# Uncomment to run | 取消注释以运行

# print("Starting a conversation with history | 开始带历史记录的对话\n")

# # First message | 第一条消息
# response1 = app.chat(
#     "Hello! Can you help me understand what machine learning is?",
#     use_history=True
# )
# print("User | 用户: Hello! Can you help me understand what machine learning is?")
# print(f"AI: {response1}\n")

# # Follow-up message (will include previous context) | 后续消息（将包含之前的上下文）
# response2 = app.chat(
#     "Can you give me a simple example?",
#     use_history=True
# )
# print("User | 用户: Can you give me a simple example?")
# print(f"AI: {response2}\n")

# # Clear history | 清除历史
# app.clear_history()

print("Uncomment the code above to test conversations with history | 取消注释上面的代码以测试带历史记录的对话")

## 8. Summary and Next Steps | 总结和后续步骤

### What We've Built | 我们构建了什么

This notebook demonstrated a comprehensive bilingual AI application with:

本笔记本演示了一个全面的双语AI应用，包括：

1. **Model Management** | **模型管理**
   - Fetching and caching OpenAI models | 获取和缓存OpenAI模型
   - Exporting to CSV and Excel formats | 导出为CSV和Excel格式
   - Model-specific parameter recommendations | 特定模型的参数推荐

2. **Prompt Management** | **提示词管理**
   - Saving and loading prompts | 保存和加载提示词
   - Metadata tracking | 元数据跟踪
   - Collection import/export | 集合导入/导出

3. **Chat Integration** | **聊天集成**
   - Direct API calls with custom parameters | 使用自定义参数直接调用API
   - Using saved prompts | 使用保存的提示词
   - Conversation history management | 对话历史管理

### Potential Enhancements | 潜在的增强功能

- Add streaming responses | 添加流式响应
- Implement prompt templates with variables | 实现带变量的提示词模板
- Add support for image generation | 添加图像生成支持
- Create a web interface | 创建Web界面
- Add usage tracking and cost estimation | 添加使用跟踪和成本估算
- Implement prompt versioning | 实现提示词版本控制

### Resources | 资源

- [OpenAI API Documentation](https://platform.openai.com/docs)
- [OpenAI Python Library](https://github.com/openai/openai-python)
- [Pandas Documentation](https://pandas.pydata.org/docs/)
- [openpyxl Documentation](https://openpyxl.readthedocs.io/)

---

**Thank you for using this tutorial! | 感谢使用本教程！**

Feel free to modify and extend this code for your own projects.

欢迎修改和扩展此代码以用于您自己的项目。